# Exercise 3.6 — Concentration of Purity (Lévy's Lemma)

**Chapter 3: Haar Ensembles** &nbsp;|&nbsp; *Section 3.6: Concentration of Measure*

---

## Motivation

**Concentration of measure** is the mathematical mechanism behind quantum typicality: for Haar-random states in high-dimensional Hilbert spaces, observables such as entanglement entropy and purity are sharply concentrated around their mean values.  Lévy's lemma provides the quantitative bound — the probability of a deviation $\epsilon$ from the mean decays exponentially in the Hilbert space dimension $D$.  This exercise applies Lévy's lemma to the purity, showing that in multi-qubit systems, virtually every random state has nearly the same subsystem purity.

## Preliminaries / Toolbox

Before diving into the solution, recall the following concepts:

**1. Purity:** The purity of a state $\rho$ is $\gamma = \mathrm{Tr}(\rho^2)$. For pure states $\gamma = 1$, and for maximally mixed states $\gamma = 1/D$.

**2. Haar Measure:** The uniform distribution over the unitary group $U(D)$. Integrals over polynomials of $U_{ij}$ and $\bar{U}_{kl}$ can be evaluated using Weingarten calculus.

**3. Lévy's Lemma:** Given a Lipschitz-continuous function $f: \mathbb{S}^{2D-1} \to \mathbb{R}$ on the unit hypersphere, the probability that $f(|\psi\rangle)$ deviates from its median by more than $\epsilon$ decays exponentially with dimension $D$: $\Pr(|f - \bar{f}| \geq \epsilon) \leq 2 \exp\left(- \frac{C D \epsilon^2}{\eta^2}\right)$.

**4. Partial Trace:** The reduced density matrix of subsystem $A$ is $\rho_A = \mathrm{Tr}_B(|\psi\rangle\langle\psi|)$, where $|\psi\rangle$ is a bipartite state on $\mathcal{H}_A \otimes \mathcal{H}_B$.



## Exercise Statement

The purity $\gamma = \mathrm{Tr}(\rho_A^2)$ of a Haar-random state has mean $\mathbb{E}[\gamma] = (m+n)/(mn+1)$.  The function $f(\psi) = \mathrm{Tr}(\rho_A^2)$ is Lipschitz with constant $\eta_f = 4$.

**(a)** Using Lévy's lemma, write the concentration bound $\Pr(|\gamma - \mathbb{E}[\gamma]| > \epsilon)$.

**(b)** For an equal bipartition ($m = n = 2^{N/2}$), find the minimum $N$ such that $\Pr(\gamma > \mathbb{E}[\gamma] + 0.01) < 10^{-6}$.

## Solution

### Part (a): Lévy's lemma bound

**Lévy's lemma** states: if $f: S^{2D-1} \to \mathbb{R}$ is Lipschitz with constant $\eta_f$, then for Haar-random $|\psi\rangle$:

$$
\Pr\!\left(|f(\psi) - \mathbb{E}[f]| > \epsilon\right) \leq 2\exp\!\left(-\frac{c\, D\, \epsilon^2}{\eta_f^2}\right),
$$

where $c > 0$ is a universal constant of order unity and $D = mn$ is the total Hilbert space dimension.

For the purity with $\eta_f = 4$:

$$
\boxed{\Pr\!\left(|\gamma - \mathbb{E}[\gamma]| > \epsilon\right) \leq 2\exp\!\left(-\frac{c\, D\, \epsilon^2}{16}\right).}
$$

The concentration is **exponentially tight** in $D$: doubling the number of qubits squares the Hilbert space dimension and drives the failure probability to zero exponentially fast.

### Part (b): Critical qubit number

For the equal bipartition $m = n = 2^{N/2}$, we have $D = 2^N$.  The condition is:

$$
2\exp\!\left(-\frac{c \cdot 2^N \cdot 10^{-4}}{16}\right) < 10^{-6}.
$$

Taking logarithms (with $c \approx 1$):

$$
\frac{2^N \cdot 10^{-4}}{16} > \ln(2 \times 10^6) \approx 14.5.
$$

$$
2^N > \frac{16 \times 14.5}{10^{-4}} \approx 2.3 \times 10^6.
$$

Since $2^{21} \approx 2.1 \times 10^6$ and $2^{22} \approx 4.2 \times 10^6$:

$$
\boxed{N \geq 22 \text{ qubits}.}
$$

With just 22 qubits (a 22-qubit Hilbert space of dimension $\sim 4 \times 10^6$), the probability that the purity deviates by more than 0.01 from its mean is less than one in a million.  This extreme concentration is why typicality arguments are so powerful in quantum many-body physics.

---
## Symbolic Verification (SymPy)

In [ ]:
import sympy as sp

d_A, d_B = sp.symbols('d_A d_B', positive=True, integer=True)
D = d_A * d_B

# Page formula for expected purity of subsystem A
# E[Tr(rho_A^2)] = (d_A + d_B) / (d_A * d_B + 1)
E_purity = (d_A + d_B) / (D + 1)
print(f'E[purity_A] = {E_purity}')

# For equal bipartition d_A = d_B = d
d = sp.Symbol('d', positive=True, integer=True)
E_equal = E_purity.subs([(d_A, d), (d_B, d)])
print(f'Equal bipartition: E[purity] = {sp.simplify(E_equal)}')

# Large d limit: E[purity] -> 2d/(d^2+1) -> 2/d
print(f'Large d limit: {sp.limit(E_equal, d, sp.oo)} (maximally mixed = 1/d)')

# Numerical values
for n in [2, 4, 8, 16]:
    val = float(2*n / (n**2 + 1))
    print(f'  d={n:3d}: E[purity] = {val:.6f}, 1/d = {1/n:.6f}')

---
## Numerical Verification

We verify concentration at small $N$ by sampling Haar-random states and checking the purity distribution.

In [ ]:
import numpy as np
from scipy.stats import unitary_group

np.random.seed(42)

print(f"{'N':>3s}  {'m':>4s}  {'D':>6s}  {'E[γ] (MC)':>10s}  {'(m+n)/(mn+1)':>14s}  {'std(γ)':>8s}")
print(f"{'─'*3:>3s}  {'─'*4:>4s}  {'─'*6:>6s}  {'─'*10:>10s}  {'─'*14:>14s}  {'─'*8:>8s}")

for N in [4, 6, 8]:
    m = 2**(N//2)
    n = m
    D = m * n
    E_gamma_pred = (m + n) / (m*n + 1)
    
    n_samples = min(5000, max(500, 100000//D))
    purities = []
    for _ in range(n_samples):
        psi = unitary_group.rvs(D)[:, 0]
        rho = np.outer(psi, psi.conj()).reshape(m, n, m, n)
        rho_A = np.trace(rho, axis1=1, axis2=3)
        purities.append(np.trace(rho_A @ rho_A).real)
    
    purities = np.array(purities)
    E_gamma = np.mean(purities)
    std_gamma = np.std(purities)
    
    print(f"{N:3d}  {m:4d}  {D:6d}  {E_gamma:10.4f}  {E_gamma_pred:14.4f}  {std_gamma:8.4f}")
    assert abs(E_gamma - E_gamma_pred) < 0.02

print("\nConcentration sharpens with D: std(γ) → 0. ✓")

---
## Visual Pedagogical Proof

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import unitary_group

np.random.seed(42)
purities_8 = []
purities_16 = []

for _ in range(500):
    psi8 = unitary_group.rvs(8)[:, 0]
    rho8 = psi8.reshape(2, 4) @ psi8.reshape(2, 4).conj().T
    purities_8.append(np.trace(rho8 @ rho8).real)
    
    psi16 = unitary_group.rvs(16)[:, 0]
    rho16 = psi16.reshape(4, 4) @ psi16.reshape(4, 4).conj().T
    purities_16.append(np.trace(rho16 @ rho16).real)

plt.rcParams['figure.dpi'] = 150
plt.figure(figsize=(7, 4.5))
plt.hist(purities_8, bins=30, alpha=0.6, color='crimson', label='D=8 (Subsystem d=2)')
plt.hist(purities_16, bins=30, alpha=0.6, color='royalblue', label='D=16 (Subsystem d=4)')

plt.xlabel(r"Subsystem Purity $\gamma$")
plt.ylabel("Frequency")
plt.title("Concentration of Measure: Subsystem Purity")
plt.axvline(1/2, color='crimson', linestyle='dashed', label='Max Mixed d=2')
plt.axvline(1/4, color='royalblue', linestyle='dashed', label='Max Mixed d=4')
plt.legend()
plt.grid(True, ls="--", alpha=0.5)
plt.tight_layout()
plt.show()